# Detecção preditiva de fraudes em períodos de alta movimentação

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, confusion_matrix, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

FAST = os.environ.get("FINANCE_BI_FAST_TEST") == "1"
CSV = Path(os.environ.get("FINANCE_BI_PAYSIM_CSV", "data/raw/PS_20174392719_1491204439457_log.csv"))
OUT = Path(os.environ.get("FINANCE_BI_OUTPUT_DIR", "outputs")) / "03_fraudes"
OUT.mkdir(parents=True,exist_ok=True)
assert CSV.is_file(), f"CSV não encontrado: {CSV}"
cols = ["step","type","amount","isFraud","isFlaggedFraud"]
dtypes = {"step":"int32","type":"category","amount":"float32","isFraud":"int8","isFlaggedFraud":"int8"}
df = pd.read_csv(CSV,usecols=cols,dtype=dtypes)
df = df.dropna(subset=cols).loc[df["amount"].ge(0)].copy()
assert set(df["isFraud"].unique()).issubset({0,1})
assert set(df["isFlaggedFraud"].unique()).issubset({0,1})
print("Transações PaySim:",len(df),"fraudes:",int(df["isFraud"].sum()), "período simulado (etapas):",df["step"].min(),df["step"].max())
print("Taxa de fraude:",float(df["isFraud"].mean()))
display(df.groupby("type",observed=True)["isFraud"].agg(["sum","count","mean"]).sort_values("mean",ascending=False))

In [ ]:
steps = np.sort(df["step"].unique())
assert len(steps)>=30,"Poucas etapas para separar cronologicamente."
corte_treino = steps[int(len(steps)*.60)]
corte_val = steps[int(len(steps)*.80)]
raw_train=df.loc[df["step"]<corte_treino].copy()
raw_val=df.loc[(df["step"]>=corte_treino)&(df["step"]<corte_val)].copy()
raw_test=df.loc[df["step"]>=corte_val].copy()
for nome,parte in [('treino',raw_train),('validação',raw_val),('teste',raw_test)]:
    print(nome,len(parte),"fraudes",int(parte["isFraud"].sum()),"taxa",round(float(parte["isFraud"].mean()),6))
    assert parte["isFraud"].nunique()==2, f"Sem ambas as classes em {nome}."
raw_train["hora"]=(raw_train["step"]-1)%24
volume_por_hora=raw_train.groupby("hora")["amount"].sum().reindex(range(24),fill_value=0)
picos=set(volume_por_hora.nlargest(max(1,int(np.ceil(24*.25)))).index.tolist())
print("Horários de pico definidos APENAS pelo período de treino:",sorted(picos))
print("Campos proibidos para modelagem:", ["oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest","isFlaggedFraud","nameOrig","nameDest"])

In [ ]:
tipos=sorted(str(v) for v in df["type"].unique())
feature_cols=["log_valor","hora_seno","hora_cosseno","horario_pico"]+["tipo_"+t for t in tipos]
def transformar(parte):
    hora=((parte["step"].to_numpy()-1)%24).astype(int)
    values = {"log_valor":np.log1p(parte["amount"].to_numpy()),"hora_seno":np.sin(2*np.pi*hora/24),"hora_cosseno":np.cos(2*np.pi*hora/24),"horario_pico":np.isin(hora,list(picos)).astype(int)}
    for t in tipos:values["tipo_"+t] = parte["type"].eq(t).to_numpy(dtype=int)
    return pd.DataFrame(values,index=parte.index).astype('float32')[feature_cols]
X_val=transformar(raw_val); y_val=raw_val["isFraud"].to_numpy()
X_test=transformar(raw_test); y_test=raw_test["isFraud"].to_numpy()
positivos=raw_train.loc[raw_train["isFraud"].eq(1)]
negativos=raw_train.loc[raw_train["isFraud"].eq(0)]
max_neg=2500 if FAST else 350_000
neg_amostra=negativos.sample(n=min(len(negativos),max_neg),random_state=42)
subtreino=pd.concat([positivos,neg_amostra]).sort_index()
X_train=transformar(subtreino)
y_train=subtreino["isFraud"].to_numpy()
# Ponderação corrige a probabilidade de seleção dos negativos na subamostragem.
peso_negativos=len(negativos)/max(1,len(neg_amostra))
pesos=np.where(y_train==0,peso_negativos,1.).astype(float)
assert np.isfinite(X_train.to_numpy()).all()
print("Amostra de treino:",len(subtreino),"| peso dos negativos:",round(peso_negativos,3))

In [ ]:
modelos = {
    "Random Forest":RandomForestClassifier(n_estimators=50 if FAST else 160,max_depth=10,min_samples_leaf=8,random_state=42,n_jobs=2),
    "XGBoost":XGBClassifier(n_estimators=50 if FAST else 180,max_depth=4,learning_rate=.05,subsample=.85,colsample_bytree=.9,objective="binary:logistic",eval_metric="logloss",random_state=42,n_jobs=2),
}
resultados=[]; probs_test={}; modelos_treinados={}
for nome,modelo in modelos.items():
    modelo.fit(X_train,y_train,sample_weight=pesos)
    prob_val=modelo.predict_proba(X_val)[:,1]
    p,r,thr=precision_recall_curve(y_val,prob_val)
    f1s=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
    limiar=float(thr[int(np.argmax(f1s))]) if len(thr) else .5
    probs=modelo.predict_proba(X_test)[:,1]
    pred=(probs>=limiar).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    resultados.append({"modelo":nome,"limiar_definido_na_validacao":limiar,"ap_teste":average_precision_score(y_test,probs),"roc_auc_teste":roc_auc_score(y_test,probs),"precisao_teste":precision_score(y_test,pred,zero_division=0),"recall_teste":recall_score(y_test,pred,zero_division=0),"f1_teste":f1_score(y_test,pred,zero_division=0),"tp":int(tp),"fp":int(fp),"fn":int(fn),"tn":int(tn)})
    probs_test[nome]=probs;modelos_treinados[nome]=modelo
relatorio=pd.DataFrame(resultados).sort_values("ap_teste",ascending=False)
print("Comparação na janela de teste intocada (taxa original de fraude preservada):")
display(relatorio)
print("Referência: prevalência de fraude no teste =",round(float(y_test.mean()),6))

In [ ]:
melhor=str(relatorio.iloc[0]["modelo"])
prob=probs_test[melhor]
limiar=float(relatorio.iloc[0]["limiar_definido_na_validacao"])
resultado_teste=raw_test[["step","type","amount","isFraud","isFlaggedFraud"]].copy()
resultado_teste["probabilidade_modelo"] = prob
resultado_teste["predicao_modelo"] = (prob>=limiar).astype(int)
resultado_teste["horario_pico"] = X_test["horario_pico"].to_numpy(dtype=int)
print("Desempenho por período de pico; 'horário_pico' é volume treinado, não pico futuro conhecido:")
display(resultado_teste.groupby("horario_pico").agg(transacoes=("isFraud","size"),fraudes_reais=("isFraud","sum"),alertas=("predicao_modelo","sum"),fraudes_sinalizadas_pela_regra=("isFlaggedFraud","sum")))
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(np.log1p(raw_test.loc[raw_test["isFraud"].eq(0),"amount"]),bins=35,alpha=.6,label="Legítima",density=True)
axes[0].hist(np.log1p(raw_test.loc[raw_test["isFraud"].eq(1),"amount"]),bins=35,alpha=.6,label="Fraude",density=True)
axes[0].legend();axes[0].set(xlabel="log(1+valor)",title="Distribuição de valores no teste")
axes[1].hist(prob[y_test==0],bins=35,alpha=.6,density=True,label="Legítima")
axes[1].hist(prob[y_test==1],bins=35,alpha=.6,density=True,label="Fraude")
axes[1].set(xlabel="Score do modelo",title="Separação de classes no teste");axes[1].legend()
plt.tight_layout();plt.show()

In [ ]:
try:
    import shap
    pequeno=X_test.sample(n=min(150,len(X_test)),random_state=42)
    explainer=shap.TreeExplainer(modelos_treinados[melhor])
    shap_values=explainer(pequeno)
    if len(shap_values.values.shape)==3:
        valores=shap_values.values[:,:,1]
    else:
        valores=shap_values.values
    importancia=pd.DataFrame({"variavel":pequeno.columns,"media_abs_shap":np.mean(np.abs(valores),axis=0)}).sort_values("media_abs_shap",ascending=False)
    display(importancia)
    shap.summary_plot(valores,pequeno,show=False)
    plt.tight_layout();plt.show()
except (ImportError,Exception) as exc:
    print("SHAP indisponível ou incompatível neste ambiente:",repr(exc))

In [ ]:
relatorio.to_csv(OUT/"metricas_fraude_holdout.csv",index=False)
resultado_teste.to_csv(OUT/"scores_fraude_teste.csv",index=False)
print("Arquivos:",*[str(p) for p in sorted(OUT.glob('*.csv'))],sep="\n")
print("AUC-PR é central diante do desbalanceamento. O dataset é sintético; desempenho não demonstra detecção real em produção.")